[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PacktPublishing/Data-Strategy-for-LLMs/blob/main/chapter_07/Jupyter_Notebooks/Chapter_7_Notebook.ipynb)

**Click the badge above to run this notebook in Google Colab (no local setup needed).**


## Setup

**This chapter uses local fine-tuning with Hugging Face Transformers and PyTorch.**

1. Run the book-wide setup once from the repo root: `bash setup/setup_mac.sh` (macOS/Linux) or `powershell -ExecutionPolicy Bypass -File setup/setup_windows.ps1` (Windows). This creates the `data_strategy_env` environment and the **"Python (Data Strategy Book)"** Jupyter kernel.
2. Select the **"Python (Data Strategy Book)"** kernel (top-right). If missing: Command Palette -> "Developer: Reload Window".

The next cell installs any missing local fine-tuning packages **into the running kernel** and locates the Chapter 7 datasets.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import sys
import subprocess
from pathlib import Path

def _install(pkg):
    for cmd in ([sys.executable, "-m", "pip", "install", pkg, "--quiet"],
                [sys.executable, "-m", "pip", "install", pkg, "--user", "--quiet"],
                [sys.executable, "-m", "pip", "install", pkg, "--break-system-packages", "--quiet"]):
        try:
            subprocess.run(cmd, check=True, capture_output=True, text=True)
            return True
        except subprocess.CalledProcessError:
            continue
    return False

for _pkg in ("torch", "transformers", "peft", "ipywidgets"):
    if not _install(_pkg):
        print(f"WARNING: could not install {_pkg} (restart the kernel and re-run this cell)")

# Locate the repository root from local notebooks, VS Code, or Colab.
repo_root = Path.cwd()
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / "chapter_07" / "datasets" / "sft_train.jsonl").exists():
        repo_root = _p
        break

chapter_dir = repo_root / "chapter_07"
dataset_dir = chapter_dir / "datasets"
dataset_dir.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "distilgpt2"  # small enough for CPU demos; switch to another causal LM if you have GPU memory.
SFT_MODEL_DIR = dataset_dir / "sft_model"

print(f"Repo root: {repo_root}")
print(f"Training data: {dataset_dir / 'sft_train.jsonl'}")
print(f"Base model: {BASE_MODEL}")
print("Setup complete.")

Repo root: d:\Code\Data-Strategy-for-LLMs
Training data: d:\Code\Data-Strategy-for-LLMs\chapter_07\datasets\sft_train.jsonl
Base model: distilgpt2
Setup complete.


## Applying Supervised Fine-Tuning

Read the same minimal, "failure-shaped" SFT examples, fine-tune a small local causal language model, and save the resulting model for later evaluation. This keeps the chapter runnable after hosted OpenAI fine-tuning was deprecated for some organizations.

In [ ]:
import json
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

train_path = dataset_dir / "sft_train.jsonl"
valid_path = dataset_dir / "sft_valid.jsonl"

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

train_examples = load_jsonl(train_path)
valid_examples = load_jsonl(valid_path)
print(f"Loaded {len(train_examples)} train and {len(valid_examples)} validation examples")

def split_messages(messages):
    assistant_index = max(index for index, message in enumerate(messages) if message["role"] == "assistant")
    return messages[:assistant_index], messages[assistant_index]["content"]

def render_messages(messages):
    rendered = []
    for message in messages:
        rendered.append(f"{message['role'].title()}:\n{message['content'].strip()}")
    return "\n\n".join(rendered)

class ChatSFTDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=256):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        prompt_messages, answer = split_messages(self.examples[index]["messages"])
        prompt_text = render_messages(prompt_messages) + "\n\nAssistant:\n"
        full_text = prompt_text + answer.strip() + self.tokenizer.eos_token

        encoded = self.tokenizer(
            full_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)
        labels = input_ids.clone()

        prompt_len = len(self.tokenizer(prompt_text, add_special_tokens=False)["input_ids"])
        labels[:min(prompt_len, self.max_length)] = -100
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)

train_loader = DataLoader(ChatSFTDataset(train_examples, tokenizer), batch_size=2, shuffle=True)
valid_loader = DataLoader(ChatSFTDataset(valid_examples, tokenizer), batch_size=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(3):
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(**batch)
        outputs.loss.backward()
        optimizer.step()
        total_loss += outputs.loss.item()
    print(f"Epoch {epoch + 1}/3 -- train loss: {total_loss / len(train_loader):.4f}")

model.eval()
valid_loss = 0.0
with torch.no_grad():
    for batch in valid_loader:
        valid_loss += model(**batch).loss.item()
print(f"Validation loss: {valid_loss / len(valid_loader):.4f}")

# Chapter 9 loads this directory for the base-vs-SFT comparison.
SFT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(SFT_MODEL_DIR))
tokenizer.save_pretrained(str(SFT_MODEL_DIR))

metadata_path = dataset_dir / "sft_model_info.json"
metadata_path.write_text(
    json.dumps({"base_model": BASE_MODEL, "model_dir": str(SFT_MODEL_DIR)}, indent=2),
    encoding="utf-8",
)
print(f"Saved SFT model to {SFT_MODEL_DIR}")
print(f"Saved model metadata to {metadata_path}")

Loaded 10 train and 3 validation examples


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1/3 -- train loss: 4.3591
Epoch 2/3 -- train loss: 3.6334
Epoch 3/3 -- train loss: 3.1148
Validation loss: 3.7618


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved SFT model to d:\Code\Data-Strategy-for-LLMs\chapter_07\datasets\sft_model
Saved model metadata to d:\Code\Data-Strategy-for-LLMs\chapter_07\datasets\sft_model_info.json


In [ ]:
# Smoke-test the same saved SFT model artifact that Chapter 9 loads later.
sample_messages, expected_answer = split_messages(valid_examples[0]["messages"])
sample_prompt = render_messages(sample_messages) + "\n\nAssistant:\n"

sft_tokenizer = AutoTokenizer.from_pretrained(str(SFT_MODEL_DIR))
sft_model = AutoModelForCausalLM.from_pretrained(str(SFT_MODEL_DIR))
sft_model.eval()

inputs = sft_tokenizer(sample_prompt, return_tensors="pt")
with torch.no_grad():
    generated = sft_model.generate(
        **inputs,
        max_new_tokens=90,
        do_sample=False,
        pad_token_id=sft_tokenizer.eos_token_id,
    )

print("Prompt:\n", sample_prompt)
print("Expected answer:\n", expected_answer)
print("Generated answer:\n", sft_tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Prompt:
 System:
You are a policy-aware support assistant. If context is insufficient, say 'Information not available' and ask a clarifying question. Output: Summary, Next steps, Risks.

User:
Latency spiked after enabling new cache. Disabling cache restored performance. Summarize and propose next steps.

Assistant:

Expected answer:
 Summary
Latency increased after enabling the cache; disabling it restored performance.

Next steps
- Compare cache config (TTL/keying/eviction) before vs after
- Inspect hit rate and backend saturation during spike
- Re-enable via canary and isolate the specific setting

Risks
Re-enabling without isolating the setting can recreate the incident.
Generated answer:
 - Cache restored performance.

- Cache restored performance.


Risks
- Cache restored performance.

- Cache restored performance.


Risks
- Cache restored performance.

- Cache restored performance.


Risks
- Cache restored performance.

- Cache restored performance.


Risks
- Cache restored perf

## Applying LoRA in practice

This example mirrors the SFT workflow, but shows how LoRA is applied as a **parameter‑efficient method**. Data preparation is assumed to be identical to SFT. The important part is not the parameters. It is the deployment model: the adapter can be attached, detached, or replaced without disturbing the base model.

In [ ]:
%pip install -q peft transformers torch ipywidgets

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
import torch
import os
from pathlib import Path

BASE = "distilgpt2"  # small model -- runs on CPU

tokenizer = AutoTokenizer.from_pretrained(BASE)
tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(BASE)

# --- Configure LoRA adapter (rank, alpha, and which layers to target are explicit) ---
lora_config = LoraConfig(
    r=16,                          # low-rank dimension
    lora_alpha=32,                 # scaling factor
    target_modules=["c_attn"],     # inject into attention projection only
    lora_dropout=0.05,
    bias="none",
)

# ATTACH: wrap base model with LoRA -- base weights are frozen, only adapter params are trained
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# --- Train adapter to learn one fact ---
FACT = "The capital of France is Paris."
train_enc = tokenizer(FACT, return_tensors="pt")
labels = train_enc["input_ids"].clone()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
model.train()
for step in range(50):
    optimizer.zero_grad()
    loss = model(**train_enc, labels=labels).loss
    loss.backward()
    optimizer.step()
    if (step + 1) % 10 == 0:
        print(f"Step {step + 1}/50 -- loss: {loss.item():.4f}")
model.eval()

prompt = tokenizer("The capital of France is", return_tensors="pt")

# Stop at "." so generation ends at the sentence boundary
stop_token_id = tokenizer.encode(".", add_special_tokens=False)[0]
gen_kwargs = dict(max_new_tokens=10, eos_token_id=stop_token_id)

# ATTACH: adapter active -- model has learned the fact
print("\n=== Adapter ATTACHED ===")
with torch.no_grad():
    out = model.generate(**prompt, **gen_kwargs)
print(tokenizer.decode(out[0]))

# DETACH: bypass adapter -- base weights are untouched, original behaviour restored
model.disable_adapter_layers()
print("\n=== Adapter DETACHED (base model) ===")
with torch.no_grad():
    out = model.generate(**prompt, **gen_kwargs)
print(tokenizer.decode(out[0]))
model.enable_adapter_layers()

# SAVE: persist adapter to repo so Chapter 9 can load it for evaluation
adapter_dir = Path("../datasets/lora_adapter")
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(adapter_dir))
print(f"\nAdapter saved to {adapter_dir}")
print("Adapter files:", os.listdir(adapter_dir))
print(f"Total size: {sum(f.stat().st_size for f in adapter_dir.iterdir()) / 1024:.1f} KB")

# REPLACE: reload adapter onto a fresh base to verify it works
fresh_base = AutoModelForCausalLM.from_pretrained(BASE)
replaced_model = PeftModel.from_pretrained(fresh_base, str(adapter_dir))
replaced_model.eval()
print("\n=== Adapter REPLACED -- inference with reloaded adapter ===")
with torch.no_grad():
    out = replaced_model.generate(**prompt, **gen_kwargs)
print(tokenizer.decode(out[0]))

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 20500.81it/s]


trainable params: 294,912 || all params: 82,207,488 || trainable%: 0.3587
Step 10/50 — loss: 2.6535
Step 20/50 — loss: 1.9481
Step 30/50 — loss: 1.8068
Step 40/50 — loss: 0.4869


Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.


Step 50/50 — loss: 0.0311

=== Adapter ATTACHED ===
The capital of France is Paris.

=== Adapter DETACHED (base model) ===
The capital of France is the capital of the French Republic.

Adapter saved to: <temp dir>
Adapter files: ['adapter_config.json', 'adapter_model.safetensors', 'README.md']


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 15231.61it/s]
Setting `pad_token_id` to `eos_token_id`:13 for open-end generation.



=== Adapter REPLACED — inference with reloaded adapter ===
The capital of France is Paris.
